# Track 2: ClipCap Baseline (worker 2 of 3)

Trains the ClipCap baseline (frozen GPT-2, trained mapping network) on Track 1's RN50x4 features.
The work is split over **3 Google accounts running at the same time**, one notebook each:

| | worker 0 (owner account) | worker 1 | worker 2 |
|---|---|---|---|
| Smoke tests + benchmark | yes (skipped once passed) | - | - |
| Mini sweep, 3 epochs, seed 0 | transformer p40 | transformer p10 | MLP p10 |
| Pick the baseline | each worker waits for all 3 sweep scores, then picks the best val CIDEr | | |
| Final baseline, 10 epochs | seed 0 | seed 1 | seed 2 |
| Collect results, copy to team folder | yes (waits for all 3 seeds) | - | - |

**How to run:** pick a GPU runtime, then Runtime > Run all. After a disconnect, just Run all again:
finished work is skipped and training resumes from its last checkpoint (saved every 10 minutes and at every epoch end).

- Shared folder for all workers: `My Drive/sc4001_track2/` (owned by worker 0's account, shared as Editor with the others).
- Features are raw; every prefix is L2-normalised in `normalise_prefix` (model code).
- Decoding: greedy, max 30 tokens, stop at `.`. Tune and select on val only; the test cell is off.

## Worker

In [ ]:
# set by the file name: track_2_worker0/1/2.ipynb - do not change
WORKER = 2
SWEEP_SPECS = [
    {'mapping': 'transformer', 'prefix_length': 40},  # worker 0 (the official ClipCap transformer config)
    {'mapping': 'transformer', 'prefix_length': 10},  # worker 1
    {'mapping': 'mlp', 'prefix_length': 10},          # worker 2
]
SEEDS = [0, 1, 2]  # final baseline: worker n trains seed n
SWEEP_EPOCHS = 3
FINAL_EPOCHS = 10
BEST_SPEC_OVERRIDE = None  # leave None; if the team overrides the choice, set the SAME value in all 3 notebooks
assert WORKER in (0, 1, 2)
print('worker', WORKER, '| sweep config:', SWEEP_SPECS[WORKER], '| final seed:', SEEDS[WORKER])

# Setup

## Installs

In [ ]:
!pip -q install transformers pycocoevalcap inflect
!apt-get -qq update > /dev/null && apt-get -qq install -y default-jre > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

!test -d /content/CLIP_prefix_caption || git clone -q https://github.com/rmokady/CLIP_prefix_caption /content/CLIP_prefix_caption

## Imports

In [ ]:
import os
import sys
import json
import pickle
import random
import shutil
import subprocess
import time
import zipfile
import warnings
import logging

import nltk
import inflect
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import GPT2TokenizerFast, get_linear_schedule_with_warmup

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

# ClipCap's train.py imports AdamW from transformers, removed in newer versions.
# Setting transformers.AdamW no longer works on recent versions, so patch the import line once.
TRAIN_PY = '/content/CLIP_prefix_caption/train.py'
OLD_IMPORT = 'from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup'
NEW_IMPORT = ('from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup\n'
              'from torch.optim import AdamW')
with open(TRAIN_PY) as fh:
    train_src = fh.read()
if OLD_IMPORT in train_src:
    with open(TRAIN_PY, 'w') as fh:
        fh.write(train_src.replace(OLD_IMPORT, NEW_IMPORT))
    print('patched ClipCap train.py AdamW import')
sys.path.append('/content/CLIP_prefix_caption')
from train import ClipCaptionPrefix, MappingType

warnings.filterwarnings('ignore')
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)

# stop Java warnings from corrupting pycocoevalcap output (tokenizer, METEOR)
os.environ['JAVA_TOOL_OPTIONS'] = '-Xlog:disable'

for pkg in ['punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

device = 'cuda'
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type'
print('using:', torch.cuda.get_device_name(0))

## Mount GDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob

def find_project_root(name='sc4001_nndl_grp_project'):
    # usual place: a folder or shortcut at the top of My Drive
    candidates = ['/content/drive/MyDrive/' + name]
    # a shortcut placed inside another folder, or Colab's shortcut mirror
    candidates += sorted(glob.glob('/content/drive/MyDrive/*/' + name))
    candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/' + name))
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'phase_1', 'track_1')):
            return c
    raise FileNotFoundError(
        name + ' is not in your My Drive. Colab cannot see "Shared with me". In Google Drive: Shared with me > '
        'right-click ' + name + ' > Organize > Add shortcut > My Drive, then rerun this cell.')

PROJECT_ROOT = find_project_root()
print('project folder:', PROJECT_ROOT)
TRACK1_ROOT = os.path.join(PROJECT_ROOT, 'phase_1', 'track_1')
SHARED_TRACK2 = os.path.join(PROJECT_ROOT, 'phase_1', 'track_2')  # final results are copied here at the end

# Everything Track 2 writes goes to sc4001_track2, owned by worker 0's account and shared
# with the other two accounts as Editor (each adds a shortcut to it in My Drive).
def find_track2_root(name='sc4001_track2'):
    candidates = ['/content/drive/MyDrive/' + name]
    candidates += sorted(glob.glob('/content/drive/MyDrive/*/' + name))
    candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/' + name))
    for c in candidates:
        if os.path.isdir(c):
            return c
    if WORKER == 0:
        os.makedirs(candidates[0])
        return candidates[0]
    raise FileNotFoundError(
        name + ' not found. It must be shared with this account (Editor), then: Shared with me > '
        'right-click ' + name + ' > Organize > Add shortcut > My Drive, then rerun this cell.')

ROOT = find_track2_root()

# Track 1 inputs (read only)
DATA_DIR = os.path.join(TRACK1_ROOT, 'data')
FEAT_DIR = os.path.join(TRACK1_ROOT, 'features')
EVAL_DIR = os.path.join(TRACK1_ROOT, 'eval')
TRACK1_RESULTS = os.path.join(TRACK1_ROOT, 'results')
PRETRAINED_CKPT = os.path.join(TRACK1_ROOT, 'models', 'clipcap_transformer_coco_pretrained.pt')

# Track 2 outputs; evaluate() writes its .eval.json files to RESULTS_DIR
RESULTS_DIR = os.path.join(ROOT, 'results')
MODELS_DIR = os.path.join(ROOT, 'models')
RUNS_DIR = os.path.join(ROOT, 'runs')
CACHE_DIR = os.path.join(ROOT, 'cache')
LOCAL = '/content/track2_local'  # local disk for throwaway smoke-test runs

os.makedirs(LOCAL, exist_ok=True)
if WORKER == 0:
    for d in [RESULTS_DIR, MODELS_DIR, RUNS_DIR, CACHE_DIR]:
        os.makedirs(d, exist_ok=True)

for p in [os.path.join(FEAT_DIR, 'coco_rn50x4_train.pkl'), os.path.join(FEAT_DIR, 'coco_rn50x4_val.pkl'),
          PRETRAINED_CKPT, os.path.join(EVAL_DIR, 'chair_gt_val.pkl'), os.path.join(EVAL_DIR, 'synonyms.txt'),
          os.path.join(DATA_DIR, 'karpathy_index.json'),
          os.path.join(TRACK1_RESULTS, 'clipcap_pretrained_seed0_val.json'),
          os.path.join(TRACK1_RESULTS, 'clipcap_pretrained_seed0_val.eval.json')]:
    assert os.path.exists(p), 'missing Track 1 file: ' + p
print('Track 1 files found')
print('Track 2 shared folder:', ROOT)
if WORKER != 0:
    assert os.path.isdir(os.path.join(ROOT, 'runs')), 'sc4001_track2 looks empty - is this the shared folder from worker 0?'

# show saved progress, so after switching accounts you can see the checkpoints were found
for name in sorted(os.listdir(RUNS_DIR)):
    run_dir = os.path.join(RUNS_DIR, name)
    if os.path.exists(os.path.join(run_dir, 'mapping_final.pt')) or os.path.exists(os.path.join(run_dir, 'run_log.json')):
        status = 'finished'
    elif os.path.exists(os.path.join(run_dir, 'last.pt')):
        saved = time.strftime('%Y-%m-%d %H:%M', time.gmtime(os.path.getmtime(os.path.join(run_dir, 'last.pt'))))
        status = f'in progress, last checkpoint {saved} UTC (will resume)'
    else:
        status = 'started, no checkpoint yet'
    print(f'  {name:<40} {status}')

# Helpers

Same code as `phase_1/track_2/track_2.py`.

## Constants

In [ ]:
MAX_TOKENS = 30  # team rule: greedy, max 30 tokens, stop at '.'

## General helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def write_json(obj, path):
    # tmp then rename, so a disconnect never leaves a half file
    with open(path + '.tmp', 'w') as fh:
        json.dump(obj, fh)
    os.replace(path + '.tmp', path)


def read_json(path):
    with open(path) as fh:
        return json.load(fh)


def save_torch(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)

## Data

In [ ]:
def load_features(pkl_path):
    # ClipCap pkl: {'clip_embedding': [n, 640] float16, 'captions': [{image_id, caption, clip_embedding}]}
    with open(pkl_path, 'rb') as fh:
        data = pickle.load(fh)

    features = data['clip_embedding']
    captions = data['captions']
    row_image = [None] * features.shape[0]

    for c in captions:
        row_image[c['clip_embedding']] = c['image_id']

    assert all(i is not None for i in row_image), 'feature row with no caption'
    return {'features': features, 'captions': captions, 'row_image': row_image}


def subset_images(split_data, n_images, seed=0):
    # random subset of images (with all their captions), rows renumbered
    n = split_data['features'].shape[0]
    rows = sorted(random.Random(seed).sample(range(n), n_images))
    new_row = {}

    for j in range(len(rows)):
        new_row[rows[j]] = j

    captions = []
    for c in split_data['captions']:
        if c['clip_embedding'] in new_row:
            captions.append({'image_id': c['image_id'], 'caption': c['caption'],
                             'clip_embedding': new_row[c['clip_embedding']]})

    row_image = [split_data['row_image'][r] for r in rows]
    return {'features': split_data['features'][rows], 'captions': captions, 'row_image': row_image}


def tokenize_captions(captions, tokenizer, cache_path=None):
    # list of GPT-2 token id lists, one per caption (no EOS, same as ClipCap)
    texts = [c['caption'] for c in captions]

    if cache_path is not None and os.path.exists(cache_path):
        with open(cache_path, 'rb') as fh:
            cached = pickle.load(fh)
        if cached['texts'] == texts:
            return cached['tokens']
        print('token cache does not match captions, rebuilding')

    tokens = tokenizer(texts, add_special_tokens=False)['input_ids']
    assert all(len(t) > 0 for t in tokens), 'empty caption'

    if cache_path is not None:
        with open(cache_path + '.tmp', 'wb') as fh:
            pickle.dump({'texts': texts, 'tokens': tokens}, fh)
        os.replace(cache_path + '.tmp', cache_path)
    return tokens


def max_seq_len(token_lists):
    # ClipCap's rule: mean + 10 std, capped at the longest caption
    lengths = torch.tensor([len(t) for t in token_lists]).float()
    return min(int(lengths.mean() + lengths.std() * 10), int(lengths.max()))


class CaptionDataset(Dataset):
    # one item per caption: (token ids, feature row)
    def __init__(self, split_data, token_lists, max_len):
        assert len(token_lists) == len(split_data['captions'])
        self.rows = [c['clip_embedding'] for c in split_data['captions']]
        self.tokens = [t[:max_len] for t in token_lists]
        self.features = split_data['features']
        self.max_len = max_len

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        return self.tokens[i], self.rows[i]


def collate(batch):
    # pad to the longest caption in the batch; padded labels are -100 (ignored)
    longest = max(len(t) for t, _ in batch)
    tokens = torch.zeros(len(batch), longest, dtype=torch.long)
    labels = torch.full((len(batch), longest), -100, dtype=torch.long)

    for i, (t, _) in enumerate(batch):
        tokens[i, :len(t)] = torch.tensor(t)
        labels[i, :len(t)] = torch.tensor(t)

    rows = torch.tensor([r for _, r in batch], dtype=torch.long)
    return tokens, labels, rows

## Model

In [ ]:
def build_model(mapping, prefix_length, clip_length=None, num_layers=8, prefix_size=640):
    # ClipCap model with GPT-2 frozen; needs CLIP_prefix_caption on sys.path
    from train import ClipCaptionPrefix, MappingType

    if mapping == 'mlp':
        mapping_type = MappingType.MLP
    elif mapping == 'transformer':
        mapping_type = MappingType.Transformer
    else:
        raise ValueError(mapping)

    if clip_length is None:
        clip_length = prefix_length

    model = ClipCaptionPrefix(prefix_length, clip_length=clip_length, prefix_size=prefix_size,
                              num_layers=num_layers, mapping_type=mapping_type)
    # no grads for GPT-2 weights; grads still flow through it to the prefix
    model.gpt.requires_grad_(False)
    return model


def n_mapping_params(model):
    return sum(p.numel() for p in model.clip_project.parameters())


def load_pretrained_checkpoint(model, path):
    # official checkpoint: 0 missing keys, only stale GPT-2 attention buffers unexpected
    state = torch.load(path, map_location='cpu')
    result = model.load_state_dict(state, strict=False)
    assert len(result.missing_keys) == 0, result.missing_keys
    for key in result.unexpected_keys:
        assert key.endswith('.attn.bias') or key.endswith('.attn.masked_bias'), key
    return model


def normalise_prefix(features):
    # raw CLIP features -> float32, unit length per row
    prefix = features.float()
    return prefix / prefix.norm(dim=-1, keepdim=True)


def prefix_embeddings(model, features):
    prefix = normalise_prefix(features)
    return model.clip_project(prefix).reshape(prefix.shape[0], model.prefix_length, -1)


def caption_loss(model, tokens, labels, features, reduction='mean'):
    # teacher-forced NLL of the caption tokens, same targets as ClipCap's train.py:
    # position prefix_length-1 predicts token 0, and so on
    prefix_embeds = prefix_embeddings(model, features)
    text_embeds = model.gpt.transformer.wte(tokens)
    inputs = torch.cat((prefix_embeds.to(text_embeds.dtype), text_embeds), dim=1)

    # run lm_head only on the positions that predict caption tokens
    hidden = model.gpt.transformer(inputs_embeds=inputs).last_hidden_state
    p = model.prefix_length
    logits = model.gpt.lm_head(hidden[:, p - 1:-1])

    return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), labels.reshape(-1),
                           ignore_index=-100, reduction=reduction)

## Greedy decoding (same algorithm as Track 1's greedy_decode_batch)

In [ ]:
def stop_token_id(tokenizer):
    return tokenizer.encode('.')[0]


@torch.no_grad()
def greedy_decode_batch(model, tokenizer, features, device, max_tokens=MAX_TOKENS):
    # features: [B, 640] raw CLIP vectors
    stop_id = stop_token_id(tokenizer)
    n = features.shape[0]
    embeds = prefix_embeddings(model, features.to(device))

    token_ids = [[] for _ in range(n)]
    finished = torch.zeros(n, dtype=torch.bool, device=device)

    for _ in range(max_tokens):
        logits = model.gpt(inputs_embeds=embeds).logits[:, -1, :]
        next_ids = logits.argmax(dim=-1)
        next_list = next_ids.tolist()

        for b in range(n):
            # keep tokens up to and including the first stop
            if not finished[b]:
                token_ids[b].append(next_list[b])

        finished = finished | (next_ids == stop_id)
        if bool(finished.all()):
            break

        next_embed = model.gpt.transformer.wte(next_ids).unsqueeze(1)
        embeds = torch.cat((embeds, next_embed), dim=1)

    return [tokenizer.decode(t).strip() for t in token_ids]


def decode_split(model, tokenizer, split_data, out_path, device, batch_size=64):
    # captions for every image in the split -> [{image_id, caption}], resumable
    model.eval()
    partial_path = out_path.replace('.json', '.partial.json')
    row_image = split_data['row_image']
    features = split_data['features']

    if os.path.exists(out_path):
        return read_json(out_path)

    done = {}
    if os.path.exists(partial_path):
        for p in read_json(partial_path):
            done[p['image_id']] = p['caption']

    n_rows = features.shape[0]
    starts = list(range(0, n_rows, batch_size))

    for k, start in enumerate(tqdm(starts, desc='captioning')):
        rows = [j for j in range(start, min(start + batch_size, n_rows)) if row_image[j] not in done]
        if len(rows) == 0:
            continue

        caps = greedy_decode_batch(model, tokenizer, features[rows], device)
        for i in range(len(rows)):
            done[row_image[rows[i]]] = caps[i]

        if k % 10 == 9:
            write_json([{'image_id': i, 'caption': c} for i, c in done.items()], partial_path)

    out = [{'image_id': int(row_image[j]), 'caption': done[row_image[j]]} for j in range(n_rows)]
    assert len(out) == n_rows
    write_json(out, out_path)
    if os.path.exists(partial_path):
        os.remove(partial_path)
    return out

## Training

In [ ]:
def default_config(**overrides):
    # ClipCap train.py defaults: 10 epochs, bs 40, lr 2e-5, 5000 warmup steps
    cfg = {
        'mapping': 'transformer',
        'prefix_length': 40,
        'clip_length': None,      # None -> same as prefix_length
        'num_layers': 8,
        'epochs': 10,
        'train_steps': None,      # if set, train exactly this many steps instead of `epochs`
        'batch_size': 40,
        'lr': 2e-5,
        'warmup_steps': 5000,
        'seed': 0,
        'amp': True,              # bf16 if the GPU has it, else fp16 with loss scaling
        'ckpt_minutes': 10,       # save a resume checkpoint this often (also at every epoch end)
        'log_every': 500,
    }
    for k in overrides:
        assert k in cfg, f'unknown config key {k}'
        cfg[k] = overrides[k]
    return cfg


def config_name(cfg):
    return f"{cfg['mapping']}_p{cfg['prefix_length']}"


def model_from_config(cfg):
    return build_model(cfg['mapping'], cfg['prefix_length'], cfg['clip_length'], cfg['num_layers'])


# keys that must match when resuming a run
RUN_KEYS = ['mapping', 'prefix_length', 'clip_length', 'num_layers', 'epochs', 'train_steps',
            'batch_size', 'lr', 'warmup_steps', 'seed']


def total_train_steps(cfg, n_train):
    if cfg['train_steps'] is not None:
        return cfg['train_steps']
    return cfg['epochs'] * (n_train // cfg['batch_size'])


def amp_dtype(device):
    # bf16 only on GPUs with native support (A100, L4); T4 uses fp16 with loss scaling
    if device == 'cuda' and torch.cuda.get_device_capability(0)[0] >= 8:
        return torch.bfloat16
    return torch.float16


def epoch_batches(n_items, batch_size, seed, epoch):
    # same shuffle for a given (seed, epoch), so a resumed run sees the same batches
    g = torch.Generator().manual_seed(seed * 100003 + epoch)
    perm = torch.randperm(n_items, generator=g).tolist()
    n_steps = n_items // batch_size  # drop_last, as in ClipCap
    return [perm[i * batch_size:(i + 1) * batch_size] for i in range(n_steps)]


@torch.no_grad()
def dataset_loss(model, dataset, features_gpu, device, cfg, batch_size=200):
    # mean NLL per caption token over a whole dataset
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    total = 0.0
    count = 0

    for tokens, labels, rows in loader:
        tokens, labels = tokens.to(device), labels.to(device)
        with torch.autocast('cuda', dtype=amp_dtype(device), enabled=cfg['amp'] and device == 'cuda'):
            loss_sum = caption_loss(model, tokens, labels, features_gpu[rows.to(device)], reduction='sum')
        total += float(loss_sum)
        count += int((labels != -100).sum())

    return total / count


def train_run(cfg, train_set, val_set, run_dir, device, max_steps=None, deadline=None, log=print):
    """Train one config. Resumes from run_dir/last.pt; returns (model, run_log).

    max_steps stops early (smoke tests); such runs are not marked finished.
    deadline (a time.time() value) saves a checkpoint and stops, for job time limits;
    the run is not marked finished and resumes from that checkpoint next time.
    """
    os.makedirs(run_dir, exist_ok=True)
    state_path = os.path.join(run_dir, 'last.pt')
    final_path = os.path.join(run_dir, 'mapping_final.pt')
    log_path = os.path.join(run_dir, 'run_log.json')

    set_seed(cfg['seed'])
    model = model_from_config(cfg)

    # finished earlier: load and return
    if os.path.exists(final_path) and os.path.exists(log_path):
        saved = torch.load(final_path, map_location='cpu')
        model.clip_project.load_state_dict(saved['clip_project'])
        log('run already finished:', run_dir)
        return model.to(device), read_json(log_path)

    model = model.to(device)
    bs = cfg['batch_size']
    steps_per_epoch = len(train_set) // bs
    total_steps = total_train_steps(cfg, len(train_set))
    n_epochs = -(-total_steps // steps_per_epoch)  # ceil: the last epoch may be partial

    optimizer = torch.optim.AdamW(model.clip_project.parameters(), lr=cfg['lr'])
    scheduler = get_linear_schedule_with_warmup(optimizer, cfg['warmup_steps'], total_steps)
    use_amp = cfg['amp'] and device == 'cuda'
    dtype = amp_dtype(device)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and dtype == torch.float16)

    run_log = {
        'config': cfg,
        'run_dir': run_dir,
        'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu',
        'amp_dtype': str(dtype) if use_amp else 'float32',
        'mapping_params': n_mapping_params(model),
        'n_train_captions': len(train_set),
        'steps_per_epoch': steps_per_epoch,
        'total_steps': total_steps,
        'train_loss': [],    # [global_step, mean loss since last log]
        'epochs': [],        # one record per finished epoch
        'train_seconds': 0.0,
        'finished': False,
    }
    start_epoch = 0
    start_step = 0

    if os.path.exists(state_path):
        state = torch.load(state_path, map_location='cpu', weights_only=False)
        for k in RUN_KEYS:
            assert state['run_log']['config'][k] == cfg[k], f'resume config mismatch on {k}'
        model.clip_project.load_state_dict(state['clip_project'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch = state['epoch']
        start_step = state['step']
        run_log = state['run_log']
        log(f'resuming {run_dir} at epoch {start_epoch} step {start_step}')

    def save_state(epoch, step):
        save_torch({
            'clip_project': model.clip_project.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'epoch': epoch,
            'step': step,
            'run_log': run_log,
        }, state_path)

    features_gpu = train_set.features.to(device)
    val_features_gpu = val_set.features.to(device) if val_set is not None else None
    global_step = start_epoch * steps_per_epoch + start_step
    stopped_early = False

    for epoch in range(start_epoch, n_epochs):
        skip = start_step if epoch == start_epoch else 0
        batches = epoch_batches(len(train_set), bs, cfg['seed'], epoch)[skip:]
        loader = DataLoader(train_set, batch_sampler=batches, collate_fn=collate, num_workers=0)

        model.train()
        t_epoch = time.time()
        t_saved = time.time()
        loss_sum = 0.0
        loss_n = 0
        epoch_loss_sum = 0.0
        epoch_loss_n = 0
        step = skip

        bar = tqdm(loader, initial=skip, total=steps_per_epoch, desc=f'{config_name(cfg)} s{cfg["seed"]} ep{epoch}')
        for tokens, labels, rows in bar:
            tokens = tokens.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            prefix = features_gpu[rows.to(device)]

            with torch.autocast('cuda', dtype=dtype, enabled=use_amp):
                loss = caption_loss(model, tokens, labels, prefix)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            step += 1
            global_step += 1
            value = float(loss.detach())
            loss_sum += value
            loss_n += 1
            epoch_loss_sum += value
            epoch_loss_n += 1

            if global_step % cfg['log_every'] == 0:
                run_log['train_loss'].append([global_step, loss_sum / loss_n])
                bar.set_postfix(loss=f'{loss_sum / loss_n:.3f}')
                loss_sum = 0.0
                loss_n = 0

            if global_step >= total_steps:
                break  # planned end of training, possibly mid-epoch

            if max_steps is not None and global_step >= max_steps:
                stopped_early = True
                break

            if time.time() - t_saved > cfg['ckpt_minutes'] * 60:
                run_log['train_seconds'] += time.time() - t_saved
                save_state(epoch, step)
                t_saved = time.time()

            if deadline is not None and time.time() > deadline:
                run_log['train_seconds'] += time.time() - t_saved
                save_state(epoch, step)
                t_saved = time.time()
                log(f'time limit reached: saved checkpoint at epoch {epoch} step {step}')
                stopped_early = True
                break

        run_log['train_seconds'] += time.time() - t_saved
        if stopped_early:
            break

        epoch_seconds = time.time() - t_epoch
        record = {
            'epoch': epoch,
            'train_loss': epoch_loss_sum / max(epoch_loss_n, 1),
            'session_seconds': epoch_seconds,
            'session_steps': epoch_loss_n,
            'session_sec_per_step': epoch_seconds / max(epoch_loss_n, 1),
        }
        if val_set is not None:
            record['val_loss'] = dataset_loss(model, val_set, val_features_gpu, device, cfg)
        run_log['epochs'].append(record)
        log(f'epoch {epoch}: ' + ', '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}'
                                          for k, v in record.items()))
        save_state(epoch + 1, 0)
        model.train()

    if stopped_early:
        run_log['stopped_at_step'] = global_step
        return model, run_log

    run_log['finished'] = True
    save_torch({'config': cfg, 'clip_project': model.clip_project.state_dict()}, final_path)
    write_json(run_log, log_path)
    if os.path.exists(state_path):
        os.remove(state_path)  # big file, not needed once finished
    return model, run_log


def benchmark(cfg, train_set, device, n_steps=100, warmup=10):
    # time training steps for one config, nothing saved; returns sec/step and peak memory
    set_seed(cfg['seed'])
    model = model_from_config(cfg).to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.clip_project.parameters(), lr=cfg['lr'])
    use_amp = cfg['amp'] and device == 'cuda'
    dtype = amp_dtype(device)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and dtype == torch.float16)
    features_gpu = train_set.features.to(device)
    batches = epoch_batches(len(train_set), cfg['batch_size'], cfg['seed'], 0)[:n_steps + warmup]
    loader = DataLoader(train_set, batch_sampler=batches, collate_fn=collate)

    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    t0 = None
    for i, (tokens, labels, rows) in enumerate(loader):
        if i == warmup:
            if device == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
        with torch.autocast('cuda', dtype=dtype, enabled=use_amp):
            loss = caption_loss(model, tokens.to(device), labels.to(device), features_gpu[rows.to(device)])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    if device == 'cuda':
        torch.cuda.synchronize()
    sec_per_step = (time.time() - t0) / n_steps
    steps_per_epoch = len(train_set) // cfg['batch_size']
    result = {
        'config': config_name(cfg),
        'mapping_params_M': n_mapping_params(model) / 1e6,
        'sec_per_step': sec_per_step,
        'hours_per_epoch': sec_per_step * steps_per_epoch / 3600,
        'peak_mem_GB': torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0.0,
        # resume checkpoint = weights + 2 Adam moments, fp32
        'ckpt_GB': n_mapping_params(model) * 4 * 3 / 1e9,
    }
    del model, optimizer
    if device == 'cuda':
        torch.cuda.empty_cache()
    return result


def load_trained(run_dir, device):
    # rebuild a finished run's model from its mapping weights
    saved = torch.load(os.path.join(run_dir, 'mapping_final.pt'), map_location='cpu')
    model = model_from_config(saved['config'])
    model.clip_project.load_state_dict(saved['clip_project'])
    return model.to(device).eval()

## Track 1 evaluation helpers

Copied verbatim from `track_1.ipynb` ("Annotations > Helper Functions" `download`,
"CHAIR and Object List > Helper Functions" and "Evaluation > Helper Functions"), so scores match Track 1 exactly. Do not edit here.

In [ ]:
def download(url, out_dir):
    subprocess.run(['wget', '-c', '-q', '--show-progress', '-P', out_dir, url], check=True)
    return os.path.join(out_dir, os.path.basename(url))


def extract_members(zip_path, names, out_dir):
    # copy selected files out of a zip, tmp then rename so no half files
    with zipfile.ZipFile(zip_path) as z:
        lookup = {}
        for member in z.namelist():
            lookup[os.path.basename(member)] = member
        for name in names:
            dst = os.path.join(out_dir, name)
            with z.open(lookup[name]) as src, open(dst + '.tmp', 'wb') as out:
                shutil.copyfileobj(src, out)
            os.replace(dst + '.tmp', dst)


def norm_split(s):
    # restval counts as train
    if s in ('train', 'restval'):
        return 'train'
    else:
        return s

In [ ]:
SYN_URL = 'https://raw.githubusercontent.com/LisaAnne/Hallucination/master/data/synonyms.txt'
SYN_PATH = os.path.join(EVAL_DIR, 'synonyms.txt')

# small text file downloaded once
if not os.path.exists(SYN_PATH):
    print('downloading synonyms.txt...')
    download(SYN_URL, EVAL_DIR)

# each line is comma separated
# first word is the COCO category
NODE_WORDS = []
WORD_TO_NODE = {}
SYNONYMS = {}

with open(SYN_PATH) as fh:
    for line in fh:
        if line.strip() == '':
            continue

        group = []

        for w in line.split(','):
            # original file has stray spaces and one capital
            w = w.strip().lower()

            if w != '' and w not in group:
                group.append(w)

        node = group[0]
        NODE_WORDS.append(node)
        SYNONYMS[node] = group

        for w in group:
            WORD_TO_NODE[w] = node

OBJECT_WORDS = set(WORD_TO_NODE.keys())
print('categories:', len(NODE_WORDS), 'object words:', len(OBJECT_WORDS))
assert len(NODE_WORDS) == 80

# double word rules
# same as original CHAIR
DOUBLE_WORDS = {}

for w in ['motor bike', 'motor cycle', 'air plane', 'traffic light', 'street light',
          'traffic signal', 'stop light', 'fire hydrant', 'stop sign', 'parking meter',
          'suit case', 'sports ball', 'baseball bat', 'baseball glove', 'tennis racket',
          'wine glass', 'hot dog', 'cell phone', 'mobile phone', 'teddy bear', 'hair drier',
          'potted plant', 'bow tie', 'laptop computer', 'stove top oven', 'home plate',
          'train track']:

    DOUBLE_WORDS[w] = w

# baby elephant -> elephant, so it does not count as person
for w in ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'animal', 'cub']:
    DOUBLE_WORDS['baby ' + w] = w
    DOUBLE_WORDS['adult ' + w] = w

# passenger jet -> jet, so it does not count as person
for w in ['jet', 'train']:
    DOUBLE_WORDS['passenger ' + w] = w

DOUBLE_WORDS['bow tie'] = 'tie'
DOUBLE_WORDS['toilet seat'] = 'toilet'
DOUBLE_WORDS['wine glas'] = 'wine glass'

INFLECT = inflect.engine()

def singularize(word):
    # keep vocab words as is
    # inflect breaks some (bus -> bu)
    if word in OBJECT_WORDS:
        return word

    singular = INFLECT.singular_noun(word)

    # inflect returns False when the word is already singular
    if singular:
        return singular
    else:
        return word

def caption_to_objects(caption):
    # returns object words and their categories
    # one entry per mention, in order
    tokens = nltk.word_tokenize(caption.lower())
    words = []

    for t in tokens:
        words.append(singularize(t))

    # merge double words so hot dog is not also counted as dog
    merged = []
    i = 0

    while i < len(words):
        pair = ' '.join(words[i:i + 2])

        if pair in DOUBLE_WORDS:
            merged.append(DOUBLE_WORDS[pair])
            i += 2
        else:
            merged.append(words[i])
            i += 1

    # seat of a toilet is not a chair
    if 'toilet' in merged and 'seat' in merged:
        kept = []

        for w in merged:
            if w != 'seat':
                kept.append(w)

        merged = kept

    object_words = []
    node_words = []

    for w in merged:
        if w in OBJECT_WORDS:
            object_words.append(w)
            node_words.append(WORD_TO_NODE[w])

    return object_words, node_words

def build_gt_cache(split):
    # {image_id: {'seg': [objects], 'cap': [[objects] per caption]}}
    gt = {}
    n_no_seg = 0

    for r in tqdm(index, desc=split):
        if r['split'] != split:
            continue
        cap_objects = []

        for c in r['captions']:
            _, nodes = caption_to_objects(c)
            cap_objects.append(nodes)
        seg = []

        if r['image_id'] in seg_objects:
            seg = sorted(seg_objects[r['image_id']])
        else:
            n_no_seg += 1

        gt[r['image_id']] = {'seg': seg, 'cap': cap_objects}

    path = os.path.join(EVAL_DIR, f'chair_gt_{split}.pkl')

    with open(path + '.tmp', 'wb') as fh:
        pickle.dump(gt, fh)

    os.replace(path + '.tmp', path)
    print(split, ': saved', len(gt), 'images,', n_no_seg, 'with no segmentation objects')
    return gt

print('helpers ready')

In [ ]:
METRIC_ORDER = ['BLEU-4', 'METEOR', 'CIDEr', 'SPICE', 'CHAIRi', 'CHAIRs', 'avg_len', 'obj_recall']

# reference captions come from the index
if 'index' not in globals():
    with open(os.path.join(DATA_DIR, 'karpathy_index.json')) as fh:
        index = json.load(fh)
print('index loaded:', len(index), 'images')

def load_gt_objects(split):
    # GT objects per image = seg labels union objects in every human caption
    with open(os.path.join(EVAL_DIR, f'chair_gt_{split}.pkl'), 'rb') as fh:
        cache = pickle.load(fh)
    gt_objects = {}

    for image_id in cache:
        objs = set(cache[image_id]['seg'])

        for cap_objs in cache[image_id]['cap']:
            for o in cap_objs:
                objs.add(o)

        gt_objects[image_id] = objs

    return gt_objects

def load_gt_captions(split):
    # karpathy reference captions per image
    gt_caps = {}
    for r in index:
        if r['split'] == split:
            gt_caps[r['image_id']] = r['captions']
    return gt_caps

def compute_language_metrics(gt_caps, pred_caps, spice=False):
    # gt_caps: {image_id: [refs]}, pred_caps: {image_id: caption}
    gts = {}
    res = {}

    for image_id in pred_caps:
        refs = []
        for c in gt_caps[image_id]:
            refs.append({'caption': c})
        gts[image_id] = refs
        res[image_id] = [{'caption': pred_caps[image_id]}]

    # standard COCO tokenisation - lowercase, strip punctuation
    print('  tokenising...')
    tokenizer = PTBTokenizer()
    gts = tokenizer.tokenize(gts)
    res = tokenizer.tokenize(res)

    scores = {}
    print('  BLEU...')
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    scores['BLEU-4'] = float(bleu[3])
    print('  METEOR...')
    meteor, _ = Meteor().compute_score(gts, res)
    scores['METEOR'] = float(meteor)
    print('  CIDEr...')
    cider, _ = Cider().compute_score(gts, res)
    scores['CIDEr'] = float(cider)

    if spice:
        print('  SPICE (slow)...')
        spice_score, _ = Spice().compute_score(gts, res)
        scores['SPICE'] = float(spice_score)

    return scores

def compute_chair(pred_caps, gt_objects):
    # returns summary scores and one record per image
    per_image = []
    total_mentions = 0
    total_halluc = 0
    n_caps_halluc = 0
    total_words = 0
    recall_sum = 0.0
    n_recall = 0

    for image_id in pred_caps:
        caption = pred_caps[image_id]
        gt = gt_objects[image_id]
        _, mentioned = caption_to_objects(caption)

        # a mention is hallucinated if its category is not in the GT set
        halluc = []

        for o in mentioned:
            if o not in gt:
                halluc.append(o)

        total_mentions += len(mentioned)
        total_halluc += len(halluc)
        if len(halluc) > 0:
            n_caps_halluc += 1
        total_words += len(caption.split())

        # recall for one image = distinct GT objects mentioned / GT objects
        if len(gt) > 0:
            hit = 0
            mentioned_set = set(mentioned)

            for o in gt:
                if o in mentioned_set:
                    hit += 1

            recall_sum += hit / len(gt)
            n_recall += 1

        per_image.append({
            'image_id': image_id,
            'caption': caption,
            'mentioned_objects': mentioned,
            'hallucinated_objects': halluc,
            'gt_objects': sorted(gt)
        })

    n = len(pred_caps)
    scores = {}

    # CHAIRi: let h_k = 1 if mention k is hallucinated, else 0, over all K mentions pooled across captions
    # CHAIRi = (1/K) * sum_k h_k = hallucinated mentions / all mentions
    if total_mentions > 0:
        scores['CHAIRi'] = total_halluc / total_mentions
    else:
        scores['CHAIRi'] = 0.0

    # CHAIRs: let s_j = 1 if caption j has at least one hallucinated mention.
    # CHAIRs = (1/N) * sum_j s_j = hallucinating captions / all captions
    scores['CHAIRs'] = n_caps_halluc / n

    # avg_len = (sum_j words in caption j) / N
    scores['avg_len'] = total_words / n

    # obj_recall: r_j = |mentioned_j intersect GT_j| / |GT_j| for images with
    # |GT_j| > 0, then obj_recall = mean of r_j over those M images
    if n_recall > 0:
        scores['obj_recall'] = recall_sum / n_recall
    else:
        scores['obj_recall'] = 0.0

    scores['n_mentions'] = total_mentions
    scores['n_hallucinated'] = total_halluc
    return scores, per_image

def write_json(obj, path):
    with open(path + '.tmp', 'w') as fh:
        json.dump(obj, fh)
    os.replace(path + '.tmp', path)

def print_summary(result):
    print('-' * 40)
    print('file :', result['caption_file'])
    print('split:', result['split'], ' images:', result['n_images'])
    print('-' * 40)

    for m in METRIC_ORDER:
        if m in result['scores']:
            print(f'{m:<12}{result["scores"][m]:>10.4f}')
    print('-' * 40)

def evaluate(caption_file, split, spice=False):
    name = os.path.basename(caption_file)

    if name.endswith('.json'):
        name = name[:-5]

    result_path = os.path.join(RESULTS_DIR, name + '.eval.json')
    per_image_path = os.path.join(RESULTS_DIR, name + '.per_image.json')

    # reuse cached result unless SPICE is newly requested
    if os.path.exists(result_path):
        with open(result_path) as fh:
            cached = json.load(fh)

        if (not spice) or ('SPICE' in cached['scores']):
            print('cached result found, skipping recompute')
            print_summary(cached)
            return cached

    print('evaluating', name, 'on', split)
    with open(caption_file) as fh:
        preds = json.load(fh)

    # every image in the split exactly once, nothing extra
    gt_caps = load_gt_captions(split)
    pred_caps = {}

    for p in preds:
        image_id = int(p['image_id'])
        assert image_id not in pred_caps, f'duplicate image_id {image_id}'
        assert image_id in gt_caps, f'image_id {image_id} not in {split}'
        pred_caps[image_id] = p['caption']

    assert len(pred_caps) == len(gt_caps), \
        f'{len(pred_caps)} captions but {split} has {len(gt_caps)} images'

    print('  coverage ok:', len(pred_caps), 'images')

    scores = compute_language_metrics(gt_caps, pred_caps, spice)

    print('  CHAIR...')
    gt_objects = load_gt_objects(split)
    chair_scores, per_image = compute_chair(pred_caps, gt_objects)

    for k in chair_scores:
        scores[k] = chair_scores[k]

    result = {
        'caption_file': name,
        'split': split,
        'n_images': len(pred_caps),
        'spice': spice,
        'scores': scores
    }

    write_json(per_image, per_image_path)
    write_json(result, result_path)
    print('  saved', os.path.basename(result_path), 'and', os.path.basename(per_image_path))
    print_summary(result)
    return result

def aggregate_seeds(results):
    # mean and population std per metric across seed results
    agg = {}

    for m in METRIC_ORDER:
        values = []

        for r in results:
            if m in r['scores']:
                values.append(r['scores'][m])

        if len(values) == len(results) and len(values) > 0:
            arr = np.array(values)
            agg[m] = {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0))}
    return agg

def paired_bootstrap(per_image_a, per_image_b, metric, n=1000, seed=42):
    # pair records by image id so A and B are compared on the same images
    b_by_id = {}

    for rec in per_image_b:
        b_by_id[rec['image_id']] = rec

    mention_a = []
    halluc_a = []
    mention_b = []
    halluc_b = []

    for rec in per_image_a:
        other = b_by_id[rec['image_id']]
        mention_a.append(len(rec['mentioned_objects']))
        halluc_a.append(len(rec['hallucinated_objects']))
        mention_b.append(len(other['mentioned_objects']))
        halluc_b.append(len(other['hallucinated_objects']))

    mention_a = np.array(mention_a)
    halluc_a = np.array(halluc_a)
    mention_b = np.array(mention_b)
    halluc_b = np.array(halluc_b)

    def score(halluc, mention, idx):
        # CHAIRi - pools mentions: sum h / sum m over the sampled images
        if metric == 'chair_i':
            total = mention[idx].sum()

            if total == 0:
                return 0.0

            return halluc[idx].sum() / total

        # CHAIRs - fraction of sampled captions with at least one hallucination
        return (halluc[idx] > 0).mean()

    assert metric in ('chair_i', 'chair_s'), metric
    n_img = len(mention_a)
    all_idx = np.arange(n_img)
    observed = score(halluc_b, mention_b, all_idx) - score(halluc_a, mention_a, all_idx)

    # resample images with replacement - same sample for A and B (paired).
    # diff_t = metric_B(sample t) - metric_A(sample t); the 2.5th and 97.5th
    # percentiles of diff_1..diff_n give the 95% confidence interval
    rng = np.random.default_rng(seed)
    diffs = np.zeros(n)

    for t in range(n):
        idx = rng.integers(0, n_img, n_img)
        diffs[t] = score(halluc_b, mention_b, idx) - score(halluc_a, mention_a, idx)

    lo = float(np.percentile(diffs, 2.5))
    hi = float(np.percentile(diffs, 97.5))
    return float(observed), lo, hi

print('evaluation helpers ready')

# Data

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

print('loading features...')
train_data = load_features(os.path.join(FEAT_DIR, 'coco_rn50x4_train.pkl'))
val_data = load_features(os.path.join(FEAT_DIR, 'coco_rn50x4_val.pkl'))
print('train:', tuple(train_data['features'].shape), len(train_data['captions']), 'captions')
print('val:  ', tuple(val_data['features'].shape), len(val_data['captions']), 'captions')
assert train_data['features'].shape == (113287, 640)
assert val_data['features'].shape == (5000, 640)

# tokenise once; worker 0 writes the cache on Drive, the others only read it
def token_cache(name):
    path = os.path.join(CACHE_DIR, name)
    if WORKER == 0 or os.path.exists(path):
        return path
    return None

train_tokens = tokenize_captions(train_data['captions'], tokenizer, token_cache('train_tokens.pkl'))
val_tokens = tokenize_captions(val_data['captions'], tokenizer, token_cache('val_tokens.pkl'))

MAX_LEN = max_seq_len(train_tokens)
train_set = CaptionDataset(train_data, train_tokens, MAX_LEN)
val_set = CaptionDataset(val_data, val_tokens, MAX_LEN)
print('max caption tokens:', MAX_LEN, ' train captions:', len(train_set), ' val captions:', len(val_set))

# Smoke Tests

Cheap checks before any long run. Worker 0 only, and skipped once they have passed
(a `smoke_passed.json` marker, or the earlier 10k-image run's score, in the shared folder).

In [ ]:
SMOKE_MARKER = os.path.join(ROOT, 'smoke_passed.json')
earlier_small = os.path.join(RESULTS_DIR, 'clipcap_smoke_small10k_seed0_val.eval.json')
already_passed = os.path.exists(SMOKE_MARKER) or (
    os.path.exists(earlier_small) and read_json(earlier_small)['scores']['CIDEr'] > 0.3)
RUN_SMOKE = WORKER == 0 and not already_passed
print('running smoke tests' if RUN_SMOKE else 'skipping smoke tests (worker 0 only / already passed)')

## Our code reproduces Track 1's pretrained captions

Loads the official checkpoint into `build_model` and decodes 50 val images with our `greedy_decode_batch`.
They should match Track 1's caption file. This checks model building, normalisation and decoding together.
One or two mismatches can come from GPU numerics with a different batch size; many means a bug.

In [ ]:
if RUN_SMOKE:
    pretrained = build_model('transformer', 40)
    load_pretrained_checkpoint(pretrained, PRETRAINED_CKPT)
    pretrained = pretrained.to(device).eval()

    t1_caps = {}
    for p in read_json(os.path.join(TRACK1_RESULTS, 'clipcap_pretrained_seed0_val.json')):
        t1_caps[p['image_id']] = p['caption']

    check_rows = list(range(0, 5000, 100))
    ours = greedy_decode_batch(pretrained, tokenizer, val_data['features'][check_rows], device)
    n_match = 0
    for i, j in enumerate(check_rows):
        theirs = t1_caps[val_data['row_image'][j]]
        if ours[i] == theirs:
            n_match += 1
        else:
            print('MISMATCH', val_data['row_image'][j], '| ours:', ours[i], '| track 1:', theirs)

    print('matches Track 1:', n_match, 'of', len(check_rows))
    assert n_match >= 0.95 * len(check_rows)

## Loss matches ClipCap

`caption_loss` must equal ClipCap's own `model(tokens, prefix)` path. The loss on raw (unnormalised)
features is printed for information only: the transformer mapper is fairly insensitive to input scale,
so the gap is small. We still normalise everywhere so training, decoding and the checkpoint all match.

In [ ]:
if RUN_SMOKE:
    tokens, labels, rows = collate([val_set[i] for i in range(32)])
    tokens, labels = tokens.to(device), labels.to(device)
    feats = val_data['features'][rows].to(device)

    with torch.no_grad():
        ours = float(caption_loss(pretrained, tokens, labels, feats))
        out = pretrained(tokens, normalise_prefix(feats))
        logits = out.logits[:, pretrained.prefix_length - 1:-1]
        ref = float(F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.flatten(), ignore_index=-100))

        # raw features, skipping normalise_prefix
        raw_embeds = pretrained.clip_project(feats.float()).reshape(len(rows), 40, -1)
        out_raw = pretrained.gpt(inputs_embeds=torch.cat((raw_embeds, pretrained.gpt.transformer.wte(tokens)), 1))
        logits_raw = out_raw.logits[:, 39:-1]
        raw = float(F.cross_entropy(logits_raw.reshape(-1, logits_raw.shape[-1]), labels.flatten(), ignore_index=-100))

    print(f'our loss {ours:.4f}  ClipCap forward {ref:.4f}  raw features {raw:.4f}')
    assert abs(ours - ref) < 1e-3
    print('raw minus normalised loss:', round(raw - ours, 4), '(info only)')

## evaluate() in this notebook reproduces Track 1

Scores Track 1's pretrained caption file. Expect BLEU-4 31.4, METEOR 26.2, CIDEr 104.1, CHAIRi 7.6%, CHAIRs 11.2%.

In [ ]:
if RUN_SMOKE:
    check = evaluate(os.path.join(TRACK1_RESULTS, 'clipcap_pretrained_seed0_val.json'), 'val')
    assert abs(check['scores']['CIDEr'] - 1.0406) < 0.002, 'evaluation does not match Track 1 - check JAVA_TOOL_OPTIONS'
    del pretrained
    torch.cuda.empty_cache()

## Tiny overfit test

200 training images, high learning rate. Training loss should fall well below its start,
and captions for training images should look like their ground truth.

In [ ]:
if RUN_SMOKE:
    tiny = subset_images(train_data, 200, seed=0)
    tiny_tokens = tokenize_captions(tiny['captions'], tokenizer)
    tiny_set = CaptionDataset(tiny, tiny_tokens, MAX_LEN)

    tiny_cfg = default_config(mapping='transformer', prefix_length=10, epochs=5, lr=1e-4, warmup_steps=20, log_every=10)
    tiny_dir = os.path.join(LOCAL, 'tiny_overfit')
    shutil.rmtree(tiny_dir, ignore_errors=True)
    tiny_model, tiny_log = train_run(tiny_cfg, tiny_set, None, tiny_dir, device)

    first = tiny_log['train_loss'][0][1]
    last = tiny_log['train_loss'][-1][1]
    print(f'train loss {first:.3f} -> {last:.3f}')
    assert last < first - 1.0, 'loss did not fall on a tiny set'

    tiny_model.eval()
    preds = greedy_decode_batch(tiny_model, tokenizer, tiny['features'][:5], device)
    for j in range(5):
        gt = [c['caption'] for c in tiny['captions'] if c['clip_embedding'] == j][0]
        print('pred:', preds[j], '| gt:', gt)
    del tiny_model
    torch.cuda.empty_cache()

## Small-feature end-to-end run

10,000 training images, 1 epoch of the official config (transformer, prefix 40), decode all of val and evaluate.
This checks the full pipeline; the score will be well below the full baseline, but a CIDEr near 0 means a bug.

In [ ]:
if RUN_SMOKE:
    small = subset_images(train_data, 10000, seed=0)
    small_set = CaptionDataset(small, tokenize_captions(small['captions'], tokenizer), MAX_LEN)

    small_cfg = default_config(mapping='transformer', prefix_length=40, epochs=1, warmup_steps=500)
    small_dir = os.path.join(LOCAL, 'small_e2e')
    small_model, small_log = train_run(small_cfg, small_set, val_set, small_dir, device)
    print('train minutes:', round(small_log['train_seconds'] / 60, 1))

    small_caps = os.path.join(RESULTS_DIR, 'clipcap_smoke_small10k_seed0_val.json')
    decode_split(small_model, tokenizer, val_data, small_caps, device)
    small_result = evaluate(small_caps, 'val')
    assert small_result['scores']['CIDEr'] > 0.3, 'CIDEr far too low for a 10k-image run - check the pipeline'
    del small_model
    torch.cuda.empty_cache()

## Smoke tests passed

In [ ]:
if RUN_SMOKE:
    write_json({'passed': True, 'time': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}, SMOKE_MARKER)
    print('all smoke tests passed')

# Benchmark

Worker 0 only, once. Times 50 training steps per sweep config and extrapolates to a full epoch
(566k captions, bs 40, ~14k steps). `ckpt_GB` is the size of one resume checkpoint on Drive.

MLP with prefix 40 is left out of the sweep: its output layer grows with prefix length, giving about
480M parameters, too large to train in our time.

In [ ]:
BENCH_PATH = os.path.join(RESULTS_DIR, 'benchmark.json')

if WORKER == 0 and not os.path.exists(BENCH_PATH):
    bench = []
    for spec in SWEEP_SPECS:
        bench.append(benchmark(default_config(**spec), train_set, device, n_steps=50))
    write_json({'gpu': torch.cuda.get_device_name(0), 'results': bench}, BENCH_PATH)

if os.path.exists(BENCH_PATH):
    saved = read_json(BENCH_PATH)
    print('benchmark on', saved['gpu'])
    for r in saved['results']:
        print(f"{r['config']:<16} params {r['mapping_params_M']:7.1f}M  {r['sec_per_step']:.3f} s/step  "
              f"{r['hours_per_epoch']:.2f} h/epoch  peak {r['peak_mem_GB']:.1f} GB  ckpt {r['ckpt_GB']:.2f} GB")
else:
    print('no benchmark yet (worker 0 makes it)')

# Mini Sweep

Seed 0, `SWEEP_EPOCHS` (3) epochs for every config, so all get the same budget. Each worker trains its own config,
decodes val and evaluates with `spice=False`.

In [ ]:
def sweep_name(spec):
    return f"sweep_{config_name(default_config(**spec))}_e{SWEEP_EPOCHS}"

def sweep_paths(spec):
    name = sweep_name(spec)
    return {'name': name,
            'run_dir': os.path.join(RUNS_DIR, name + '_seed0'),
            'captions': os.path.join(RESULTS_DIR, f'clipcap_{name}_seed0_val.json'),
            'eval': os.path.join(RESULTS_DIR, f'clipcap_{name}_seed0_val.eval.json')}

my_spec = SWEEP_SPECS[WORKER]
paths = sweep_paths(my_spec)
print('worker', WORKER, 'sweep run:', paths['name'])

if not os.path.exists(paths['captions']):
    cfg = default_config(**my_spec, epochs=SWEEP_EPOCHS, seed=0)
    model, run_log = train_run(cfg, train_set, val_set, paths['run_dir'], device)
    decode_split(model, tokenizer, val_data, paths['captions'], device)
    del model
    torch.cuda.empty_cache()

evaluate(paths['captions'], 'val', spice=False)

# weights are not needed after scoring; captions, logs and scores are kept
weights = os.path.join(paths['run_dir'], 'mapping_final.pt')
if os.path.exists(weights):
    os.remove(weights)

## Wait for all 3 sweep results, then pick the baseline

Waits until every worker's sweep score is on Drive (checks every 2 minutes), then picks the best val CIDEr.
Every worker makes the same choice from the same files. CHAIR is shown but not used for selection.
If this waits much longer than the slowest run should take, check that the other accounts are still running.

In [ ]:
def wait_for(files, what):
    t0 = time.time()
    last_print = -1
    while True:
        os.listdir(RESULTS_DIR)  # refresh the Drive listing
        missing = [f for f in files if not os.path.exists(f)]
        if not missing:
            return
        minutes = int((time.time() - t0) / 60)
        if minutes // 10 != last_print:
            last_print = minutes // 10
            print(f'[{minutes} min] waiting for {what}:', [os.path.basename(f) for f in missing])
        time.sleep(120)

wait_for([sweep_paths(s)['eval'] for s in SWEEP_SPECS] +
         [os.path.join(sweep_paths(s)['run_dir'], 'run_log.json') for s in SWEEP_SPECS], 'sweep results')

sweep_rows = []
for spec in SWEEP_SPECS:
    pth = sweep_paths(spec)
    run_log = read_json(os.path.join(pth['run_dir'], 'run_log.json'))
    sweep_rows.append({'spec': spec, 'name': pth['name'], 'scores': read_json(pth['eval'])['scores'],
                       'val_loss': run_log['epochs'][-1]['val_loss'],
                       'train_hours': run_log['train_seconds'] / 3600})

print(f"{'config':<34}{'CIDEr':>8}{'BLEU-4':>8}{'METEOR':>8}{'CHAIRi':>8}{'CHAIRs':>8}{'val_loss':>10}{'hours':>7}")
for r in sweep_rows:
    s = r['scores']
    print(f"{r['name']:<34}{s['CIDEr'] * 100:>8.1f}{s['BLEU-4'] * 100:>8.1f}{s['METEOR'] * 100:>8.1f}"
          f"{s['CHAIRi'] * 100:>7.1f}%{s['CHAIRs'] * 100:>7.1f}%{r['val_loss']:>10.4f}{r['train_hours']:>7.2f}")

best = max(sweep_rows, key=lambda r: r['scores']['CIDEr'])
BEST_SPEC = BEST_SPEC_OVERRIDE if BEST_SPEC_OVERRIDE is not None else best['spec']
if WORKER == 0:
    write_json({'sweep_epochs': SWEEP_EPOCHS, 'selection': 'val CIDEr',
                'best_by_cider': best['spec'], 'chosen': BEST_SPEC, 'rows': sweep_rows},
               os.path.join(RESULTS_DIR, 'sweep_summary.json'))
print('best val CIDEr:', best['spec'], ' chosen baseline:', BEST_SPEC)

# Final Baseline

This worker trains seed `SEEDS[WORKER]` of the chosen config for `FINAL_EPOCHS` (10) epochs, then decodes val
and evaluates. Resumes from Drive after a disconnect. Only the mapping network is saved
(`models/clipcap_baseline_seed{n}_mapping.pt`); rebuild it with `load_trained`. The last epoch is used.

In [ ]:
seed = SEEDS[WORKER]
cfg = default_config(**BEST_SPEC, epochs=FINAL_EPOCHS, seed=seed)
run_dir = os.path.join(RUNS_DIR, f'baseline_{config_name(cfg)}_seed{seed}')
cap_path = os.path.join(RESULTS_DIR, f'clipcap_baseline_seed{seed}_val.json')
model_path = os.path.join(MODELS_DIR, f'clipcap_baseline_seed{seed}_mapping.pt')
print('worker', WORKER, 'final run:', os.path.basename(run_dir))

if WORKER == 0:
    write_json({'config': BEST_SPEC, 'epochs': FINAL_EPOCHS, 'seeds': SEEDS,
                'train_settings': default_config(**BEST_SPEC, epochs=FINAL_EPOCHS)},
               os.path.join(RESULTS_DIR, 'baseline_plan.json'))

if not (os.path.exists(cap_path) and os.path.exists(model_path)):
    model, run_log = train_run(cfg, train_set, val_set, run_dir, device)
    if not os.path.exists(model_path):
        shutil.copy(os.path.join(run_dir, 'mapping_final.pt'), model_path)
    decode_split(model, tokenizer, val_data, cap_path, device)
    print('seed', seed, 'train hours:', round(run_log['train_seconds'] / 3600, 2))
    del model
    torch.cuda.empty_cache()

evaluate(cap_path, 'val', spice=False)
print('worker', WORKER, 'done with seed', seed)

# Results and Timing (worker 0)

Worker 0 waits until all 3 seeds are scored, then reports mean ± std (population std, as in Track 1's
`aggregate_seeds`), the per-epoch loss curves and the training time per seed. Workers 1 and 2 can stop here.

In [ ]:
def seed_paths(seed):
    cfg = default_config(**BEST_SPEC, epochs=FINAL_EPOCHS, seed=seed)
    return {'eval': os.path.join(RESULTS_DIR, f'clipcap_baseline_seed{seed}_val.eval.json'),
            'log': os.path.join(RUNS_DIR, f'baseline_{config_name(cfg)}_seed{seed}', 'run_log.json')}

if WORKER != 0:
    print('worker', WORKER, 'finished. Worker 0 collects the results; you can close this session.')
else:
    wait_for([seed_paths(s)['eval'] for s in SEEDS] + [seed_paths(s)['log'] for s in SEEDS], 'all 3 seeds')

    seed_results = []
    timing = []
    curve_lines = ['seed,epoch,train_loss,val_loss']
    for seed in SEEDS:
        seed_results.append(read_json(seed_paths(seed)['eval']))
        run_log = read_json(seed_paths(seed)['log'])
        sec_per_step = [e['session_sec_per_step'] for e in run_log['epochs']]
        timing.append({'seed': seed, 'gpu': run_log['gpu'], 'train_hours': run_log['train_seconds'] / 3600,
                       'median_sec_per_step': float(np.median(sec_per_step)),
                       'val_loss_per_epoch': [e['val_loss'] for e in run_log['epochs']]})
        for e in run_log['epochs']:
            curve_lines.append(f"{seed},{e['epoch']},{e['train_loss']:.5f},{e['val_loss']:.5f}")

    # per-epoch loss curves for the results sheet / report
    with open(os.path.join(RESULTS_DIR, 'baseline_loss_curves.csv'), 'w') as fh:
        fh.write('\n'.join(curve_lines) + '\n')
    print('\n'.join(curve_lines))
    print()

    agg = aggregate_seeds(seed_results)
    reference = read_json(os.path.join(TRACK1_RESULTS, 'clipcap_pretrained_seed0_val.eval.json'))['scores']

    print('Baseline', BEST_SPEC, f'- {FINAL_EPOCHS} epochs, val, greedy, {len(SEEDS)} seeds')
    print(f"{'metric':<12}{'mean':>10}{'std':>10}{'pretrained':>12}")
    for m in METRIC_ORDER:
        if m not in agg:
            continue
        scale = 1 if m == 'avg_len' else 100
        ref = reference.get(m)
        ref_text = f'{ref * scale:>12.2f}' if ref is not None else f"{'-':>12}"
        print(f"{m:<12}{agg[m]['mean'] * scale:>10.2f}{agg[m]['std'] * scale:>10.2f}{ref_text}")

    print()
    for t in timing:
        print(f"seed {t['seed']}: {t['train_hours']:.2f} h on {t['gpu']}, {t['median_sec_per_step']:.3f} s/step")

    write_json({'config': BEST_SPEC, 'epochs': FINAL_EPOCHS, 'seeds': SEEDS, 'aggregate': agg,
                'per_seed': [r['scores'] for r in seed_results], 'timing': timing},
               os.path.join(RESULTS_DIR, 'baseline_summary.json'))

    if agg['CIDEr']['mean'] < 0.9:
        print('WARNING: CIDEr well below the pretrained reference - check for a bug before moving on')

# Copy Results to the Shared Folder (worker 0)

Copies captions, scores, summaries and the baseline mapping weights to
`sc4001_nndl_grp_project/phase_1/track_2/` so the team can use them. Checkpoints stay in `sc4001_track2`.

In [ ]:
if WORKER == 0:
    for sub in ['results', 'models']:
        src_dir = os.path.join(ROOT, sub)
        dst_dir = os.path.join(SHARED_TRACK2, sub)
        os.makedirs(dst_dir, exist_ok=True)
        n = 0
        for name in sorted(os.listdir(src_dir)):
            if name.endswith('.partial.json') or name.endswith('.tmp'):
                continue
            shutil.copy(os.path.join(src_dir, name), os.path.join(dst_dir, name))
            n += 1
        print(f'copied {n} files to {dst_dir}')

# Test Split (worker 0, run once at the very end)

Team rule: test once at the end. Leave `RUN_TEST = False` until the team has frozen every method.

In [ ]:
RUN_TEST = False

if RUN_TEST and WORKER == 0:
    test_data = load_features(os.path.join(FEAT_DIR, 'coco_rn50x4_test.pkl'))
    for seed in SEEDS:
        cfg = default_config(**BEST_SPEC, epochs=FINAL_EPOCHS, seed=seed)
        model = load_trained(os.path.join(RUNS_DIR, f'baseline_{config_name(cfg)}_seed{seed}'), device)
        cap_path = os.path.join(RESULTS_DIR, f'clipcap_baseline_seed{seed}_test.json')
        decode_split(model, tokenizer, test_data, cap_path, device)
        evaluate(cap_path, 'test', spice=True)
        del model
        torch.cuda.empty_cache()

# End of Session

In [ ]:
drive.flush_and_unmount()